## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        # 输入维度 28*28 = 784，隐藏层 100 个神经元，输出 10 个类别
        self.W1 = tf.Variable(tf.random.normal([784, 100]), name='W1')
        self.b1 = tf.Variable(tf.zeros([100]), name='b1')
        self.W2 = tf.Variable(tf.random.normal([100, 10]), name='W2')
        self.b2 = tf.Variable(tf.zeros([10]), name='b2')

    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        # 展平图像：将 [batch, 28, 28] 转为 [batch, 784]
        x = tf.reshape(x, [-1, 784])
        # 第一层：线性变换 + ReLU
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        # 第二层：线性变换，输出 logits
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 92.544 ; accuracy 0.12393333
epoch 1 : loss 87.8347 ; accuracy 0.12448333
epoch 2 : loss 84.1265 ; accuracy 0.1248
epoch 3 : loss 81.01808 ; accuracy 0.12563333
epoch 4 : loss 78.307495 ; accuracy 0.12675
epoch 5 : loss 75.86927 ; accuracy 0.12851667
epoch 6 : loss 73.63115 ; accuracy 0.13088334
epoch 7 : loss 71.545586 ; accuracy 0.1328
epoch 8 : loss 69.580475 ; accuracy 0.13483334
epoch 9 : loss 67.7183 ; accuracy 0.13725
epoch 10 : loss 65.94524 ; accuracy 0.13978334
epoch 11 : loss 64.25096 ; accuracy 0.14243333
epoch 12 : loss 62.630676 ; accuracy 0.14483333
epoch 13 : loss 61.079166 ; accuracy 0.1472
epoch 14 : loss 59.592625 ; accuracy 0.1495
epoch 15 : loss 58.16722 ; accuracy 0.15191667
epoch 16 : loss 56.799904 ; accuracy 0.154
epoch 17 : loss 55.48781 ; accuracy 0.15681666
epoch 18 : loss 54.229046 ; accuracy 0.15948333
epoch 19 : loss 53.021183 ; accuracy 0.16213334
epoch 20 : loss 51.861282 ; accuracy 0.16453333
epoch 21 : loss 50.74764 ; accuracy 0.16675
e